In [1]:
import pandas as pd
import numpy as np
import requests
import json
from collections import defaultdict

In [2]:
EMAIL = 'jmeiwb20@gmail.com'
KEY = 'mauvemouse78'
MA = '25'

In [3]:
counties_response = requests.get(f"https://aqs.epa.gov/data/api/list/countiesByState?email={EMAIL}&key={KEY}&state={MA}")
counties = counties_response.json()
counties

{'Header': [{'status': 'Success',
   'request_time': '2026-03-26T01:54:16-04:00',
   'url': 'https://aqs.epa.gov/data/api/list/countiesByState?email=jmeiwb20@gmail.com&key=mauvemouse78&state=25',
   'rows': 14}],
 'Data': [{'code': '001', 'value_represented': 'Barnstable'},
  {'code': '003', 'value_represented': 'Berkshire'},
  {'code': '005', 'value_represented': 'Bristol'},
  {'code': '007', 'value_represented': 'Dukes'},
  {'code': '009', 'value_represented': 'Essex'},
  {'code': '011', 'value_represented': 'Franklin'},
  {'code': '013', 'value_represented': 'Hampden'},
  {'code': '015', 'value_represented': 'Hampshire'},
  {'code': '017', 'value_represented': 'Middlesex'},
  {'code': '019', 'value_represented': 'Nantucket'},
  {'code': '021', 'value_represented': 'Norfolk'},
  {'code': '023', 'value_represented': 'Plymouth'},
  {'code': '025', 'value_represented': 'Suffolk'},
  {'code': '027', 'value_represented': 'Worcester'}]}

In [4]:
bc_response = requests.get(f"https://aqs.epa.gov/data/api/monitors/byCounty?email={EMAIL}&key={KEY}&param=88502&bdate=20150501&edate=20150502&state={MA}&county=001")
bc = bc_response.json()
bc

{'Header': [{'status': 'Success',
   'request_time': '2026-03-26T01:54:17-04:00',
   'url': 'https://aqs.epa.gov/data/api/monitors/byCounty?email=jmeiwb20@gmail.com&key=mauvemouse78&param=88502&bdate=20150501&edate=20150502&state=25&county=001',
   'rows': 1}],
 'Data': [{'state_code': '25',
   'county_code': '001',
   'site_number': '0002',
   'parameter_code': '88502',
   'poc': 1,
   'parameter_name': 'Acceptable PM2.5 AQI & Speciation Mass',
   'open_date': '2001-04-04',
   'close_date': None,
   'concurred_exclusions': None,
   'dominant_source': None,
   'measurement_scale': 'REGIONAL SCALE',
   'measurement_scale_def': '50 TO HUNDREDS KM',
   'monitoring_objective': 'GENERAL/BACKGROUND',
   'last_method_code': '707',
   'last_method_description': 'IMPROVE Module A with Cyclone Inlet-Teflon Filter, 2.2 sq. cm. - GRAVIMETRIC',
   'last_method_begin_date': '2001-04-04',
   'naaqs_primary_monitor': None,
   'qa_primary_monitor': None,
   'monitor_type': 'EPA',
   'networks': 'IMPROV

In [5]:
conc = pd.read_csv("annual_conc_by_monitor_2025.csv")

In [6]:
conc.head()

,State Code,County Code,Site Num,Parameter Code,POC,Latitude,Longitude,Datum,Parameter Name,Sample Duration,...,75th Percentile,50th Percentile,10th Percentile,Local Site Name,Address,State Name,County Name,City Name,CBSA Name,Date of Last Change
0,1,3,10,44201,1,30.497478,-87.880258,NAD83,Ozone,1 HOUR,...,0.054,0.045,0.029,"FAIRHOPE, Alabama","FAIRHOPE HIGH SCHOOL, 1 PIRATE DRIVE, FAIRHOPE...",Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
1,1,3,10,44201,1,30.497478,-87.880258,NAD83,Ozone,8-HR RUN AVG BEGIN HOUR,...,0.049,0.040,0.026,"FAIRHOPE, Alabama","FAIRHOPE HIGH SCHOOL, 1 PIRATE DRIVE, FAIRHOPE...",Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
2,1,3,10,44201,1,30.497478,-87.880258,NAD83,Ozone,8-HR RUN AVG BEGIN HOUR,...,0.049,0.040,0.026,"FAIRHOPE, Alabama","FAIRHOPE HIGH SCHOOL, 1 PIRATE DRIVE, FAIRHOPE...",Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
3,1,3,10,44201,1,30.497478,-87.880258,NAD83,Ozone,8-HR RUN AVG BEGIN HOUR,...,0.049,0.040,0.026,"FAIRHOPE, Alabama","FAIRHOPE HIGH SCHOOL, 1 PIRATE DRIVE, FAIRHOPE...",Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
4,1,3,10,88101,3,30.497478,-87.880258,NAD83,PM2.5 - Local Conditions,1 HOUR,...,10.000,6.000,1.000,"FAIRHOPE, Alabama","FAIRHOPE HIGH SCHOOL, 1 PIRATE DRIVE, FAIRHOPE...",Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29


In [7]:
pollutants_tested = conc[["Local Site Name", "Parameter Name", "State Name"]]


In [8]:
tested = defaultdict(dict)
for state in list(set(conc['State Name'])):
    state_df = conc[conc["State Name"] == state]
    for site in list(set(state_df[['Local Site Name']])):
        site_df = state_df[state_df["Local Site Name"] == site]
        params = set(site_df['Parameter Name'])
        tested[state][site] = params

In [9]:
tested = defaultdict(dict)
for i in list(set(conc['State Name'])):
    state_df = conc[conc["State Name"] == i]
    for j in list(set(state_df['Local Site Name'])):
        site_df = state_df[state_df["Local Site Name"] == j]
        params = set(site_df['Parameter Name'])
        tested[i][j] = params
        

In [10]:
conc.columns

Index(['State Code', 'County Code', 'Site Num', 'Parameter Code', 'POC',
       'Latitude', 'Longitude', 'Datum', 'Parameter Name', 'Sample Duration',
       'Pollutant Standard', 'Metric Used', 'Method Name', 'Year',
       'Units of Measure', 'Event Type', 'Observation Count',
       'Observation Percent', 'Completeness Indicator', 'Valid Day Count',
       'Required Day Count', 'Exceptional Data Count', 'Null Data Count',
       'Primary Exceedance Count', 'Secondary Exceedance Count',
       'Certification Indicator', 'Num Obs Below MDL', 'Arithmetic Mean',
       'Arithmetic Standard Dev', '1st Max Value', '1st Max DateTime',
       '2nd Max Value', '2nd Max DateTime', '3rd Max Value',
       '3rd Max DateTime', '4th Max Value', '4th Max DateTime',
       '1st Max Non Overlapping Value', '1st NO Max DateTime',
       '2nd Max Non Overlapping Value', '2nd NO Max DateTime',
       '99th Percentile', '98th Percentile', '95th Percentile',
       '90th Percentile', '75th Percentile', '

In [11]:
conc["Observation Count"]

0        4141
1        4328
2        4328
3        3071
4        5780
         ... 
59149     224
59150    6277
59151    6547
59152    6547
59153    4636
Name: Observation Count, Length: 59154, dtype: int64

In [12]:
tested

defaultdict(dict,
            {'Mississippi': {'Hinds CC': {'Outdoor Temperature',
               'Ozone',
               'PM2.5 - Local Conditions'},
              'Pascagoula': {'Nitric oxide (NO)',
               'Nitrogen dioxide (NO2)',
               'Outdoor Temperature',
               'Oxides of nitrogen (NOx)',
               'Ozone',
               'PM2.5 - Local Conditions',
               'SO2 max 5-min avg',
               'Sulfur dioxide'},
              'Gulfport Youth Court': {'Outdoor Temperature',
               'Ozone',
               'PM2.5 - Local Conditions'},
              'Cleveland Delta State': {'Outdoor Temperature',
               'Ozone',
               'PM2.5 - Local Conditions'},
              'Coffeeville': {'Ozone'},
              'Waveland': {'Outdoor Temperature',
               'Ozone',
               'PM2.5 - Local Conditions'},
              'Hernando': {'Outdoor Temperature',
               'Ozone',
               'PM2.5 - Local Conditions'},
   

In [13]:
tested["Utah"]

{'WASHAKIE': {'Barometric pressure',
  'Nitric oxide (NO)',
  'Nitrogen dioxide (NO2)',
  'Outdoor Temperature',
  'Oxides of nitrogen (NOx)',
  'Ozone',
  'Relative Humidity ',
  'Solar radiation',
  'Wind Direction - Resultant',
  'Wind Speed - Scalar'},
 'Lake Park': {'1,2,3-Trimethylbenzene',
  '1,2,4-Trimethylbenzene',
  '1,3,5-Trimethylbenzene',
  '1,3-Butadiene',
  '1-Butene',
  '1-Hexene',
  '1-Pentene',
  '2,2,4-Trimethylpentane',
  '2,2-Dimethylbutane',
  '2,3,4-Trimethylpentane',
  '2,3-Dimethylbutane',
  '2,3-Dimethylpentane',
  '2,4-Dimethylpentane',
  '2-Methylheptane',
  '2-Methylhexane',
  '2-Methylpentane',
  '3-Methylheptane',
  '3-Methylhexane',
  '3-Methylpentane',
  'Acetylene',
  'Barometric pressure',
  'Benzene',
  'Black Carbon PM2.5 at 880 nm',
  'Cyclohexane',
  'Cyclopentane',
  'Ethane',
  'Ethylbenzene',
  'Ethylene',
  'Isobutane',
  'Isopentane',
  'Isoprene',
  'Isopropylbenzene',
  'Methylcyclohexane',
  'Methylcyclopentane',
  'Nitric oxide (NO)',
  '

In [14]:
ozone = pd.read_csv("daily_44201_2025.csv")

C:\Users\jmei7\AppData\Local\Temp\ipykernel_18960\3688438501.py:1: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  ozone = pd.read_csv("daily_44201_2025.csv")


In [15]:
ozone.head()

,State Code,County Code,Site Num,Parameter Code,POC,Latitude,Longitude,Datum,Parameter Name,Sample Duration,...,AQI,Method Code,Method Name,Local Site Name,Address,State Name,County Name,City Name,CBSA Name,Date of Last Change
0,1,3,10,44201,1,30.497478,-87.880258,NAD83,Ozone,8-HR RUN AVG BEGIN HOUR,...,6.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,"FAIRHOPE, Alabama","FAIRHOPE HIGH SCHOOL, 1 PIRATE DRIVE, FAIRHOPE...",Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
1,1,3,10,44201,1,30.497478,-87.880258,NAD83,Ozone,8-HR RUN AVG BEGIN HOUR,...,61.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,"FAIRHOPE, Alabama","FAIRHOPE HIGH SCHOOL, 1 PIRATE DRIVE, FAIRHOPE...",Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
2,1,3,10,44201,1,30.497478,-87.880258,NAD83,Ozone,8-HR RUN AVG BEGIN HOUR,...,44.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,"FAIRHOPE, Alabama","FAIRHOPE HIGH SCHOOL, 1 PIRATE DRIVE, FAIRHOPE...",Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
3,1,3,10,44201,1,30.497478,-87.880258,NAD83,Ozone,8-HR RUN AVG BEGIN HOUR,...,44.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,"FAIRHOPE, Alabama","FAIRHOPE HIGH SCHOOL, 1 PIRATE DRIVE, FAIRHOPE...",Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
4,1,3,10,44201,1,30.497478,-87.880258,NAD83,Ozone,8-HR RUN AVG BEGIN HOUR,...,45.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,"FAIRHOPE, Alabama","FAIRHOPE HIGH SCHOOL, 1 PIRATE DRIVE, FAIRHOPE...",Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29


In [16]:
co = pd.read_csv("daily_42101_2025.csv")


C:\Users\jmei7\AppData\Local\Temp\ipykernel_18960\859686527.py:1: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  co = pd.read_csv("daily_42101_2025.csv")


In [17]:
co.head()

,State Code,County Code,Site Num,Parameter Code,POC,Latitude,Longitude,Datum,Parameter Name,Sample Duration,...,AQI,Method Code,Method Name,Local Site Name,Address,State Name,County Name,City Name,CBSA Name,Date of Last Change
0,1,73,23,42101,2,33.553056,-86.815,WGS84,Carbon monoxide,1 HOUR,...,NaN,93,INSTRUMENTAL - GAS FILTER CORRELATION CO ANALYZER,North Birmingham,"NO. B'HAM,SOU R.R., 3009 28TH ST. NO.",Alabama,Jefferson,Birmingham,"Birmingham-Hoover, AL",2025-10-01
1,1,73,23,42101,2,33.553056,-86.815,WGS84,Carbon monoxide,1 HOUR,...,NaN,93,INSTRUMENTAL - GAS FILTER CORRELATION CO ANALYZER,North Birmingham,"NO. B'HAM,SOU R.R., 3009 28TH ST. NO.",Alabama,Jefferson,Birmingham,"Birmingham-Hoover, AL",2025-10-01
2,1,73,23,42101,2,33.553056,-86.815,WGS84,Carbon monoxide,1 HOUR,...,NaN,93,INSTRUMENTAL - GAS FILTER CORRELATION CO ANALYZER,North Birmingham,"NO. B'HAM,SOU R.R., 3009 28TH ST. NO.",Alabama,Jefferson,Birmingham,"Birmingham-Hoover, AL",2025-10-01
3,1,73,23,42101,2,33.553056,-86.815,WGS84,Carbon monoxide,1 HOUR,...,NaN,93,INSTRUMENTAL - GAS FILTER CORRELATION CO ANALYZER,North Birmingham,"NO. B'HAM,SOU R.R., 3009 28TH ST. NO.",Alabama,Jefferson,Birmingham,"Birmingham-Hoover, AL",2025-10-01
4,1,73,23,42101,2,33.553056,-86.815,WGS84,Carbon monoxide,1 HOUR,...,NaN,93,INSTRUMENTAL - GAS FILTER CORRELATION CO ANALYZER,North Birmingham,"NO. B'HAM,SOU R.R., 3009 28TH ST. NO.",Alabama,Jefferson,Birmingham,"Birmingham-Hoover, AL",2025-10-01


In [18]:
no2 = pd.read_csv("daily_42602_2025.csv")

C:\Users\jmei7\AppData\Local\Temp\ipykernel_18960\3336421117.py:1: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  no2 = pd.read_csv("daily_42602_2025.csv")


In [19]:
no2.head()

,State Code,County Code,Site Num,Parameter Code,POC,Latitude,Longitude,Datum,Parameter Name,Sample Duration,...,AQI,Method Code,Method Name,Local Site Name,Address,State Name,County Name,City Name,CBSA Name,Date of Last Change
0,1,73,23,42602,1,33.553056,-86.815,WGS84,Nitrogen dioxide (NO2),1 HOUR,...,31,200,Teledyne-API Model 200EUP or T200UP - Photolyt...,North Birmingham,"NO. B'HAM,SOU R.R., 3009 28TH ST. NO.",Alabama,Jefferson,Birmingham,"Birmingham-Hoover, AL",2025-10-08
1,1,73,23,42602,1,33.553056,-86.815,WGS84,Nitrogen dioxide (NO2),1 HOUR,...,32,200,Teledyne-API Model 200EUP or T200UP - Photolyt...,North Birmingham,"NO. B'HAM,SOU R.R., 3009 28TH ST. NO.",Alabama,Jefferson,Birmingham,"Birmingham-Hoover, AL",2025-10-08
2,1,73,23,42602,1,33.553056,-86.815,WGS84,Nitrogen dioxide (NO2),1 HOUR,...,35,200,Teledyne-API Model 200EUP or T200UP - Photolyt...,North Birmingham,"NO. B'HAM,SOU R.R., 3009 28TH ST. NO.",Alabama,Jefferson,Birmingham,"Birmingham-Hoover, AL",2025-10-08
3,1,73,23,42602,1,33.553056,-86.815,WGS84,Nitrogen dioxide (NO2),1 HOUR,...,12,200,Teledyne-API Model 200EUP or T200UP - Photolyt...,North Birmingham,"NO. B'HAM,SOU R.R., 3009 28TH ST. NO.",Alabama,Jefferson,Birmingham,"Birmingham-Hoover, AL",2025-10-08
4,1,73,23,42602,1,33.553056,-86.815,WGS84,Nitrogen dioxide (NO2),1 HOUR,...,11,200,Teledyne-API Model 200EUP or T200UP - Photolyt...,North Birmingham,"NO. B'HAM,SOU R.R., 3009 28TH ST. NO.",Alabama,Jefferson,Birmingham,"Birmingham-Hoover, AL",2025-10-08


In [20]:
so2 = pd.read_csv("daily_42401_2025.csv")

C:\Users\jmei7\AppData\Local\Temp\ipykernel_18960\1945579195.py:1: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  so2 = pd.read_csv("daily_42401_2025.csv")


In [21]:
so2.head()

,State Code,County Code,Site Num,Parameter Code,POC,Latitude,Longitude,Datum,Parameter Name,Sample Duration,...,AQI,Method Code,Method Name,Local Site Name,Address,State Name,County Name,City Name,CBSA Name,Date of Last Change
0,1,73,23,42401,2,33.553056,-86.815,WGS84,Sulfur dioxide,1 HOUR,...,0.0,100,INSTRUMENTAL - ULTRAVIOLET FLUORESCENCE,North Birmingham,"NO. B'HAM,SOU R.R., 3009 28TH ST. NO.",Alabama,Jefferson,Birmingham,"Birmingham-Hoover, AL",2025-10-02
1,1,73,23,42401,2,33.553056,-86.815,WGS84,Sulfur dioxide,1 HOUR,...,4.0,100,INSTRUMENTAL - ULTRAVIOLET FLUORESCENCE,North Birmingham,"NO. B'HAM,SOU R.R., 3009 28TH ST. NO.",Alabama,Jefferson,Birmingham,"Birmingham-Hoover, AL",2025-10-02
2,1,73,23,42401,2,33.553056,-86.815,WGS84,Sulfur dioxide,1 HOUR,...,3.0,100,INSTRUMENTAL - ULTRAVIOLET FLUORESCENCE,North Birmingham,"NO. B'HAM,SOU R.R., 3009 28TH ST. NO.",Alabama,Jefferson,Birmingham,"Birmingham-Hoover, AL",2025-10-02
3,1,73,23,42401,2,33.553056,-86.815,WGS84,Sulfur dioxide,1 HOUR,...,1.0,100,INSTRUMENTAL - ULTRAVIOLET FLUORESCENCE,North Birmingham,"NO. B'HAM,SOU R.R., 3009 28TH ST. NO.",Alabama,Jefferson,Birmingham,"Birmingham-Hoover, AL",2025-10-02
4,1,73,23,42401,2,33.553056,-86.815,WGS84,Sulfur dioxide,1 HOUR,...,1.0,100,INSTRUMENTAL - ULTRAVIOLET FLUORESCENCE,North Birmingham,"NO. B'HAM,SOU R.R., 3009 28TH ST. NO.",Alabama,Jefferson,Birmingham,"Birmingham-Hoover, AL",2025-10-02


In [22]:
print(ozone.columns == co.columns)
print(ozone.columns == no2.columns)

[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True]
[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True]


In [23]:
# join on date, location (cbsa, ignore location if absent)

In [24]:
all_param = pd.concat([ozone, co, no2, so2])

In [25]:
len(all_param)

598171

In [26]:
all_param.columns

Index(['State Code', 'County Code', 'Site Num', 'Parameter Code', 'POC',
       'Latitude', 'Longitude', 'Datum', 'Parameter Name', 'Sample Duration',
       'Pollutant Standard', 'Date Local', 'Units of Measure', 'Event Type',
       'Observation Count', 'Observation Percent', 'Arithmetic Mean',
       '1st Max Value', '1st Max Hour', 'AQI', 'Method Code', 'Method Name',
       'Local Site Name', 'Address', 'State Name', 'County Name', 'City Name',
       'CBSA Name', 'Date of Last Change'],
      dtype='object')

In [27]:
set(all_param['CBSA Name'])

{'Adrian, MI',
 'Akron, OH',
 'Albany-Schenectady-Troy, NY',
 'Albuquerque, NM',
 'Allentown-Bethlehem-Easton, PA-NJ',
 'Altoona, PA',
 'Amarillo, TX',
 'Anchorage, AK',
 'Ann Arbor, MI',
 'Appleton, WI',
 'Ardmore, OK',
 'Arkadelphia, AR',
 'Asheville, NC',
 'Ashtabula, OH',
 'Athens-Clarke County, GA',
 'Atlanta-Sandy Springs-Roswell, GA',
 'Atlantic City-Hammonton, NJ',
 'Augusta-Richmond County, GA-SC',
 'Augusta-Waterville, ME',
 'Austin-Round Rock, TX',
 'Bakersfield, CA',
 'Baltimore-Columbia-Towson, MD',
 'Bangor, ME',
 'Baraboo, WI',
 'Barnstable Town, MA',
 'Bartlesville, OK',
 'Baton Rouge, LA',
 'Beaumont-Port Arthur, TX',
 'Beaver Dam, WI',
 'Bellingham, WA',
 'Bemidji, MN',
 'Bennington, VT',
 'Berlin, NH-VT',
 'Big Spring, TX',
 'Billings, MT',
 'Birmingham-Hoover, AL',
 'Bishop, CA',
 'Bismarck, ND',
 'Blacksburg-Christiansburg-Radford, VA',
 'Bloomington, IL',
 'Bloomington, IN',
 'Boise City, ID',
 'Borger, TX',
 'Boston-Cambridge-Newton, MA-NH',
 'Boulder, CO',
 'Bow

In [28]:
all_param.sort_values("Date Local").head()

,State Code,County Code,Site Num,Parameter Code,POC,Latitude,Longitude,Datum,Parameter Name,Sample Duration,...,AQI,Method Code,Method Name,Local Site Name,Address,State Name,County Name,City Name,CBSA Name,Date of Last Change
58201,12,57,81,44201,1,27.740033,-82.465146,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,37.0,47,INSTRUMENTAL - ULTRA VIOLET,Simmons Park,2401 19th Avenue Northwest,Florida,Hillsborough,Ruskin,"Tampa-St. Petersburg-Clearwater, FL",2025-11-17
47948,34,39,4,42602,2,40.641440,-74.208365,WGS84,Nitrogen dioxide (NO2),1 HOUR,...,30.0,74,INSTRUMENTAL - CHEMILUMINESCENCE,Elizabeth Lab,NJ Turnpike Interchange 13 Toll Plaza,New Jersey,Union,Elizabeth,"New York-Newark-Jersey City, NY-NJ-PA",2025-09-30
47612,8,69,11,44201,1,40.592543,-105.141122,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,25.0,199,Instrumental - Chemiluminescence API Model 265...,FORT COLLINS - WEST,3416 LA PORTE AVE.,Colorado,Larimer,Fort Collins,"Fort Collins, CO",2025-11-07
43962,26,163,99,42101,1,42.295824,-83.129431,WGS84,Carbon monoxide,8-HR RUN AVG END HOUR,...,2.0,54,INSTRUMENTAL - NONDISPERSIVE INFRARED,TRINITY ST MARKS - Gordy Howe International Br...,9191 W FORT ST (Detroit-Trinity),Michigan,Wayne,Detroit,"Detroit-Warren-Dearborn, MI",2025-09-17
25832,6,61,3,44201,1,38.935680,-121.099590,NAD83,Ozone,8-HR RUN AVG BEGIN HOUR,...,18.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,Auburn-Atwood,"11645 Atwood Street, Auburn",California,Placer,Auburn,"Sacramento--Roseville--Arden-Arcade, CA",2025-10-23


In [29]:
print(all_param.iloc[:, 10:].head())

  Pollutant Standard  Date Local   Units of Measure Event Type  \
0  Ozone 8-hour 2015  2025-02-28  Parts per million        NaN   
1  Ozone 8-hour 2015  2025-03-01  Parts per million        NaN   
2  Ozone 8-hour 2015  2025-03-02  Parts per million        NaN   
3  Ozone 8-hour 2015  2025-03-03  Parts per million        NaN   
4  Ozone 8-hour 2015  2025-03-04  Parts per million        NaN   

   Observation Count  Observation Percent  Arithmetic Mean  1st Max Value  \
0                  1                  6.0         0.006000          0.006   
1                 17                100.0         0.050824          0.058   
2                 17                100.0         0.038882          0.047   
3                 17                100.0         0.042765          0.048   
4                 17                100.0         0.046000          0.049   

   1st Max Hour   AQI  Method Code                             Method Name  \
0            23   6.0           87  INSTRUMENTAL - ULTRA VIOLE

In [39]:
choose = ["Date Local", 'Latitude', 'Longitude', 'Sample Duration',
         'Parameter Name', 'Observation Count', "Observation Percent", 'Arithmetic Mean', '1st Max Value', '1st Max Hour',
          "State Name", "County Name", "City Name", "CBSA Name"]


In [36]:
all_param[choose].head()

,Latitude,Longitude,Sample Duration,Parameter Name,State Name,Date Local,Observation Count,Observation Percent,Arithmetic Mean,1st Max Value,1st Max Hour,County Name,City Name,CBSA Name,Date of Last Change
0,30.497478,-87.880258,8-HR RUN AVG BEGIN HOUR,Ozone,Alabama,2025-02-28,1,6.0,0.006000,0.006,23,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
1,30.497478,-87.880258,8-HR RUN AVG BEGIN HOUR,Ozone,Alabama,2025-03-01,17,100.0,0.050824,0.058,10,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
2,30.497478,-87.880258,8-HR RUN AVG BEGIN HOUR,Ozone,Alabama,2025-03-02,17,100.0,0.038882,0.047,10,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
3,30.497478,-87.880258,8-HR RUN AVG BEGIN HOUR,Ozone,Alabama,2025-03-03,17,100.0,0.042765,0.048,9,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
4,30.497478,-87.880258,8-HR RUN AVG BEGIN HOUR,Ozone,Alabama,2025-03-04,17,100.0,0.046000,0.049,9,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29


In [37]:
all_param.sort_values("CBSA Name").head()

,State Code,County Code,Site Num,Parameter Code,POC,Latitude,Longitude,Datum,Parameter Name,Sample Duration,...,AQI,Method Code,Method Name,Local Site Name,Address,State Name,County Name,City Name,CBSA Name,Date of Last Change
124833,26,91,7,44201,1,41.995568,-83.946559,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,51.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,"6792 RAISIN CENTER HWY, LENAWEE CO.RD.COMM.OWN...",6792 RAISEN CTR HWY (TECUMSEH),Michigan,Lenawee,Tecumseh,"Adrian, MI",2025-11-14
124832,26,91,7,44201,1,41.995568,-83.946559,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,74.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,"6792 RAISIN CENTER HWY, LENAWEE CO.RD.COMM.OWN...",6792 RAISEN CTR HWY (TECUMSEH),Michigan,Lenawee,Tecumseh,"Adrian, MI",2025-11-14
124831,26,91,7,44201,1,41.995568,-83.946559,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,40.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,"6792 RAISIN CENTER HWY, LENAWEE CO.RD.COMM.OWN...",6792 RAISEN CTR HWY (TECUMSEH),Michigan,Lenawee,Tecumseh,"Adrian, MI",2025-11-14
124830,26,91,7,44201,1,41.995568,-83.946559,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,32.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,"6792 RAISIN CENTER HWY, LENAWEE CO.RD.COMM.OWN...",6792 RAISEN CTR HWY (TECUMSEH),Michigan,Lenawee,Tecumseh,"Adrian, MI",2025-11-14
124829,26,91,7,44201,1,41.995568,-83.946559,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,47.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,"6792 RAISIN CENTER HWY, LENAWEE CO.RD.COMM.OWN...",6792 RAISEN CTR HWY (TECUMSEH),Michigan,Lenawee,Tecumseh,"Adrian, MI",2025-11-14


In [33]:
all_param[all_param["CBSA Name"] == 'Daphne-Fairhope-Foley, AL']

,State Code,County Code,Site Num,Parameter Code,POC,Latitude,Longitude,Datum,Parameter Name,Sample Duration,...,AQI,Method Code,Method Name,Local Site Name,Address,State Name,County Name,City Name,CBSA Name,Date of Last Change
0,1,3,10,44201,1,30.497478,-87.880258,NAD83,Ozone,8-HR RUN AVG BEGIN HOUR,...,6.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,"FAIRHOPE, Alabama","FAIRHOPE HIGH SCHOOL, 1 PIRATE DRIVE, FAIRHOPE...",Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
1,1,3,10,44201,1,30.497478,-87.880258,NAD83,Ozone,8-HR RUN AVG BEGIN HOUR,...,61.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,"FAIRHOPE, Alabama","FAIRHOPE HIGH SCHOOL, 1 PIRATE DRIVE, FAIRHOPE...",Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
2,1,3,10,44201,1,30.497478,-87.880258,NAD83,Ozone,8-HR RUN AVG BEGIN HOUR,...,44.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,"FAIRHOPE, Alabama","FAIRHOPE HIGH SCHOOL, 1 PIRATE DRIVE, FAIRHOPE...",Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
3,1,3,10,44201,1,30.497478,-87.880258,NAD83,Ozone,8-HR RUN AVG BEGIN HOUR,...,44.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,"FAIRHOPE, Alabama","FAIRHOPE HIGH SCHOOL, 1 PIRATE DRIVE, FAIRHOPE...",Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
4,1,3,10,44201,1,30.497478,-87.880258,NAD83,Ozone,8-HR RUN AVG BEGIN HOUR,...,45.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,"FAIRHOPE, Alabama","FAIRHOPE HIGH SCHOOL, 1 PIRATE DRIVE, FAIRHOPE...",Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178,1,3,10,44201,1,30.497478,-87.880258,NAD83,Ozone,8-HR RUN AVG BEGIN HOUR,...,39.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,"FAIRHOPE, Alabama","FAIRHOPE HIGH SCHOOL, 1 PIRATE DRIVE, FAIRHOPE...",Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
179,1,3,10,44201,1,30.497478,-87.880258,NAD83,Ozone,8-HR RUN AVG BEGIN HOUR,...,51.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,"FAIRHOPE, Alabama","FAIRHOPE HIGH SCHOOL, 1 PIRATE DRIVE, FAIRHOPE...",Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
180,1,3,10,44201,1,30.497478,-87.880258,NAD83,Ozone,8-HR RUN AVG BEGIN HOUR,...,23.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,"FAIRHOPE, Alabama","FAIRHOPE HIGH SCHOOL, 1 PIRATE DRIVE, FAIRHOPE...",Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29
181,1,3,10,44201,1,30.497478,-87.880258,NAD83,Ozone,8-HR RUN AVG BEGIN HOUR,...,31.0,87,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,"FAIRHOPE, Alabama","FAIRHOPE HIGH SCHOOL, 1 PIRATE DRIVE, FAIRHOPE...",Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL",2025-10-29


In [41]:
all_param[choose].to_csv("all_parameters_daily.csv", index=False)